# LLM-Driven Config Generation

> **Note:** This notebook requires **PR #4092** to be merged into Ludwig, or `pip install ludwig>=0.14`.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ludwig-ai/ludwig/blob/main/examples/llm_config_generation/llm_config_generation.ipynb)

This notebook demonstrates Ludwig's config generation feature: describe your ML task in plain English and receive a fully validated Ludwig configuration dictionary. The LLM maps your column names to Ludwig feature types, selects an appropriate architecture, and returns a config that has already been validated against Ludwig's Pydantic schema.

**What you need:**
- An Anthropic API key (stored as `ANTHROPIC_API_KEY`) **or** an OpenAI API key (`OPENAI_API_KEY`)
- Ludwig ≥ 0.14 (provides `ludwig.config_generation`)

Run the install cell below to get started.

In [ ]:
# Install dependencies
# For the Anthropic backend (Claude models):
!pip install "ludwig>=0.14" anthropic --quiet
# For the OpenAI backend (GPT models), also run:
# !pip install openai --quiet

## Setup

Set your API key. In Google Colab you can store it under **Secrets** (the key icon in the left sidebar) and retrieve it with `userdata.get()`. Outside Colab, set the `ANTHROPIC_API_KEY` environment variable before launching the notebook.

In [ ]:
import os

# --- Colab: read from Colab Secrets ---
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("Loaded ANTHROPIC_API_KEY from Colab Secrets.")
except Exception:
    # Outside Colab: make sure the variable is already set in the environment,
    # or uncomment and fill in the line below:
    # os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
    if os.environ.get("ANTHROPIC_API_KEY"):
        print("ANTHROPIC_API_KEY found in environment.")
    else:
        print("WARNING: ANTHROPIC_API_KEY not set. Set it before running generation cells.")

# Choose your model:
CLAUDE_MODEL = "claude-sonnet-4-20250514"   # Anthropic
# GPT_MODEL = "gpt-4o"                      # OpenAI (set OPENAI_API_KEY instead)

In [ ]:
# NOTE: ludwig.config_generation is provided by PR #4092 (ludwig>=0.14).
# If you see an ImportError, that PR has not yet been merged into your installed version.
import yaml
from ludwig.config_generation import generate_config

## Example 1: Tabular classification (churn prediction)

We describe a binary classification task over a tabular customer dataset. The LLM will assign Ludwig feature types, choose an encoder, and produce a validated config.

In [ ]:
churn_description = (
    "I have customer data with the following columns: "
    "age (integer), annual_income (float), num_purchases (integer), "
    "days_since_last_purchase (integer), and country (string category). "
    "I want to predict churn (binary: 0 or 1). "
    "The dataset has about 50 000 rows."
)

churn_config = generate_config(
    churn_description,
    model=CLAUDE_MODEL,
    validate=True,
)

print(yaml.dump(churn_config, default_flow_style=False, sort_keys=False))

In [ ]:
# Build a small synthetic dataset matching the generated schema and train
import numpy as np
import pandas as pd
import tempfile, os

np.random.seed(42)
n = 300

synthetic_churn = pd.DataFrame({
    "age":                        np.random.randint(18, 70, n),
    "annual_income":               np.random.uniform(20_000, 150_000, n),
    "num_purchases":               np.random.randint(0, 200, n),
    "days_since_last_purchase":    np.random.randint(0, 365, n),
    "country":                     np.random.choice(["US", "UK", "DE", "FR", "CA"], n),
    "churn":                       np.random.randint(0, 2, n),
})

with tempfile.NamedTemporaryFile(suffix=".csv", delete=False, mode="w") as f:
    synthetic_churn.to_csv(f, index=False)
    churn_csv = f.name

print(synthetic_churn.head())

In [ ]:
from ludwig.api import LudwigModel

churn_model = LudwigModel(config=churn_config, logging_level="WARNING")
train_stats, _, output_dir = churn_model.train(
    dataset=churn_csv,
    output_directory=tempfile.mkdtemp(prefix="ludwig_churn_"),
)
os.unlink(churn_csv)
print(f"Training complete. Output: {output_dir}")

# Print validation accuracy
for feat, metrics in train_stats.get("validation", {}).items():
    for m, v in metrics.items():
        if isinstance(v, float):
            print(f"  {feat}/{m}: {v:.4f}")

## Example 2: Multi-task (classification + regression)

Ludwig natively supports training on multiple output targets in one pass. Here we describe a task where we want both a classification output and a regression output simultaneously.

In [ ]:
multitask_description = (
    "I have an e-commerce dataset with columns: "
    "customer_age (integer), account_tenure_days (integer), "
    "total_spend (float), product_category (category string), "
    "and region (category string). "
    "I want to simultaneously predict: "
    "1) will_churn (binary classification) and "
    "2) predicted_lifetime_value (continuous regression). "
    "Dataset size is roughly 100 000 rows."
)

multitask_config = generate_config(
    multitask_description,
    model=CLAUDE_MODEL,
    validate=True,
)

print("Output features:")
for out in multitask_config.get("output_features", []):
    print(f"  {out['name']} ({out['type']})")

print("\nFull config (YAML):")
print(yaml.dump(multitask_config, default_flow_style=False, sort_keys=False))

In [ ]:
# Synthetic multi-task dataset
np.random.seed(0)
n = 300

synthetic_multitask = pd.DataFrame({
    "customer_age":           np.random.randint(18, 75, n),
    "account_tenure_days":    np.random.randint(1, 3650, n),
    "total_spend":             np.random.uniform(10, 10_000, n),
    "product_category":        np.random.choice(["Electronics", "Clothing", "Food", "Books"], n),
    "region":                  np.random.choice(["North", "South", "East", "West"], n),
    "will_churn":              np.random.randint(0, 2, n),
    "predicted_lifetime_value": np.random.uniform(0, 5000, n),
})

with tempfile.NamedTemporaryFile(suffix=".csv", delete=False, mode="w") as f:
    synthetic_multitask.to_csv(f, index=False)
    multitask_csv = f.name

multitask_model = LudwigModel(config=multitask_config, logging_level="WARNING")
train_stats_mt, _, output_dir_mt = multitask_model.train(
    dataset=multitask_csv,
    output_directory=tempfile.mkdtemp(prefix="ludwig_multitask_"),
)
os.unlink(multitask_csv)
print(f"Multi-task training complete. Output: {output_dir_mt}")

## Example 3: Text + tabular (multimodal)

Ludwig can encode both free-text columns and structured tabular columns in the same model. Describe the mixed modality and the LLM will configure the appropriate text encoder alongside the tabular encoder.

In [ ]:
multimodal_description = (
    "I have a product review dataset. Each row has: "
    "review_text (free-form English text, typically 1-5 sentences), "
    "product_price (float), product_category (category string), "
    "and verified_purchase (binary). "
    "I want to predict star_rating, which is an integer from 1 to 5 (treat as category). "
    "There are about 200 000 rows."
)

multimodal_config = generate_config(
    multimodal_description,
    model=CLAUDE_MODEL,
    validate=True,
)

print("Input features and encoders:")
for feat in multimodal_config.get("input_features", []):
    encoder = feat.get("encoder", {}).get("type", "default")
    print(f"  {feat['name']} ({feat['type']}) — encoder: {encoder}")

print("\nFull config:")
print(yaml.dump(multimodal_config, default_flow_style=False, sort_keys=False))

## Validation in action

The `validate=True` flag (the default) runs the generated config through Ludwig's Pydantic schema before returning it. Here we intentionally pass a vague or contradictory description to observe how the validation step catches problems — and how clearer descriptions fix them.

In [ ]:
# Intentionally vague — the LLM may produce an ambiguous or invalid config.
bad_description = "predict stuff from other stuff"

try:
    bad_config = generate_config(
        bad_description,
        model=CLAUDE_MODEL,
        validate=True,
    )
    print("Config generated (LLM filled in defaults):")
    print(yaml.dump(bad_config, default_flow_style=False, sort_keys=False))
except Exception as exc:
    print(f"Validation error (expected for a vague description):\n  {exc}")
    print("\nTip: be explicit about column names, types, and target variables.")

In [ ]:
# Same task — but with a clear description. Validation passes.
clear_description = (
    "Predict house_price (continuous float) from bedrooms (integer), "
    "sqft (float), and location (string category). Dataset has 20 000 rows."
)

clear_config = generate_config(
    clear_description,
    model=CLAUDE_MODEL,
    validate=True,
)
print("Config generated and validated successfully:")
print(yaml.dump(clear_config, default_flow_style=False, sort_keys=False))

## Tips for good task descriptions

Getting a high-quality config back from `generate_config` is largely about giving the LLM enough context. Here are the key guidelines:

- **Name your columns explicitly.** `age (integer), annual_income (float)` is far better than "some user features". The LLM maps column names directly to Ludwig feature definitions.
- **State the target variable and its type.** "predict `churn` (binary: 0 or 1)" or "predict `revenue` (continuous float)" removes all ambiguity about what Ludwig's output feature should be.
- **Describe multi-output tasks explicitly.** For multi-task learning say: "simultaneously predict `price` (regression) and `category` (classification)".
- **Mention text columns and their nature.** "free-form English product review (typically 1-3 sentences)" lets the LLM choose a suitable text encoder rather than falling back to a bag-of-words default.
- **Include rough dataset size.** `~50 k rows` helps the LLM suggest appropriate model complexity and training settings.
- **Specify cardinality of category columns when known.** "country (category, ~200 unique values)" vs. "gender (binary category: M/F)" leads to different encoder choices.
- **If you have domain knowledge, share it.** "income values are right-skewed and range from 10 000 to 500 000" may improve preprocessing suggestions.

Even if the first generated config is not perfect, it provides an excellent starting point. Inspect the YAML, make targeted edits, and re-run training — far faster than writing the config from scratch.